<a href="https://colab.research.google.com/github/prathmesh796/airscope/blob/main/01_xkdr_bulk_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import requests
import io

In [2]:
from google.colab import userdata
XKDR_API_KEY=userdata.get('XKDR_API_KEY')

In [10]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

BASE = Path("/content/drive/MyDrive/AirScope")

RAW_DIR = BASE / "data" / "raw" / "xkdr"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Storage:", RAW_DIR)

Mounted at /content/drive
Storage: /content/drive/MyDrive/AirScope/data/raw/xkdr


In [12]:
BASE_URL = "https://airquality.xkdr.org"

headers = {
    "Authorization": f"Bearer {XKDR_API_KEY}"
}

In [13]:
response = requests.get(
    f"{BASE_URL}/v1/files",
    headers=headers,
    timeout=60
)

response.raise_for_status()

file_info = response.json()

files = file_info["files"]

print("Total files:", len(files))

for f in files[:5]:
    print(f)

Total files: 217
{'key': 'v1/stations.parquet', 'kind': 'stations', 'bytes': 33275}
{'key': 'v1/parameters.parquet', 'kind': 'parameters', 'bytes': 1524}
{'key': 'v1/measurements/year=2009/month=01/data.parquet', 'kind': 'measurements', 'year': 2009, 'month': 1, 'rows': 17060, 'bytes': 48363}
{'key': 'v1/measurements/year=2009/month=02/data.parquet', 'kind': 'measurements', 'year': 2009, 'month': 2, 'rows': 18525, 'bytes': 49726}
{'key': 'v1/measurements/year=2009/month=03/data.parquet', 'kind': 'measurements', 'year': 2009, 'month': 3, 'rows': 38675, 'bytes': 101728}


In [14]:
import re
import time
from tqdm.auto import tqdm

YEARS = range(2021, 2026)

# Select monthly measurement files for these years
selected_files = []

for f in files:
    key = f["key"]

    match = re.search(
        r"year=(\d{4})/month=(\d{2})/data\.parquet",
        key
    )

    if match:
        year = int(match.group(1))

        if year in YEARS:
            selected_files.append(key)

print("Files to download:", len(selected_files))

Files to download: 60


In [15]:
def download_file(key):
    url = f"{BASE_URL}/v1/files/{key}"

    output_path = RAW_DIR / key
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Skip files already downloaded
    if output_path.exists() and output_path.stat().st_size > 0:
        return "Already exists"

    with requests.get(
        url,
        headers=headers,
        stream=True,
        timeout=300
    ) as r:
        r.raise_for_status()

        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    return "Downloaded"


for key in tqdm(selected_files):
    try:
        status = download_file(key)
    except Exception as e:
        print(f"Failed: {key} | {e}")

  0%|          | 0/60 [00:00<?, ?it/s]

In [16]:
metadata_files = [
    f["key"]
    for f in files
    if f["key"] in [
        "v1/stations.parquet",
        "v1/parameters.parquet"
    ]
]

for key in metadata_files:
    download_file(key)

print("Metadata downloaded")

Metadata downloaded


In [17]:
!pip install -q pyarrow

In [18]:
sample_file = next(RAW_DIR.glob(
    "v1/measurements/year=*/month=*/data.parquet"
))

df = pd.read_parquet(sample_file)

print("File:", sample_file)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

df.head()

File: /content/drive/MyDrive/AirScope/data/raw/xkdr/v1/measurements/year=2021/month=01/data.parquet
Rows: 2940572
Columns: ['station_id', 'station_name', 'state_name', 'city_name', 'parameter_name', 'unit', 'collected_at', 'value', 'source']


,station_id,station_name,state_name,city_name,parameter_name,unit,collected_at,value,source
0,DS1010001,US Embassy - New Delhi,Delhi,New Delhi,PM2.5,µg/m³,2021-01-01 00:30:00,340.0,us_embassy
1,DS1010001,US Embassy - New Delhi,Delhi,New Delhi,PM2.5,µg/m³,2021-01-01 01:30:00,325.0,us_embassy
2,DS1010001,US Embassy - New Delhi,Delhi,New Delhi,PM2.5,µg/m³,2021-01-01 02:30:00,344.0,us_embassy
3,DS1010001,US Embassy - New Delhi,Delhi,New Delhi,PM2.5,µg/m³,2021-01-01 03:30:00,663.0,us_embassy
4,DS1010001,US Embassy - New Delhi,Delhi,New Delhi,PM2.5,µg/m³,2021-01-01 04:30:00,630.0,us_embassy
